In [ ]:
%pip install pandas
import pandas as pd
raw_df = pd.read_csv("statisctics1_Mitochondria_Raw.tsv default  node.csv")

In [ ]:
%pip install pandas requests py4cytoscape

In [ ]:
import pandas as pd, os

files = ["mitochondrion_annotations.txt"]      # add the manual and high-throughput files here
keep_methods = None    # None = keep everything; or e.g. ["manually curated", "high-throughput"]

ann = pd.concat([pd.read_csv(f, sep="\t", comment="!") for f in files if os.path.exists(f)],
                ignore_index=True)
ann.columns = ann.columns.str.strip()
ID, NAME = "Systematic Name/Complex Accession", "Gene/Complex"

ann = ann[~ann[ID].astype(str).str.startswith("CPX-")]                              # drop complexes
ann = ann[~ann["Qualifier"].astype(str).str.contains("NOT", case=False, na=False)]  # drop negatives
print(ann["Method"].value_counts())

if keep_methods:
    ann = ann[ann["Method"].isin(keep_methods)]

genes = (ann[[ID, NAME]].dropna(subset=[ID]).drop_duplicates(ID)
         .rename(columns={ID: "orf", NAME: "symbol"}))
genes["orf"] = genes["orf"].astype(str).str.strip()
genes["symbol"] = genes["symbol"].fillna(genes["orf"])
genes.to_csv("mito_genes.tsv", sep="\t", index=False)
print(len(genes), "unique genes")
assert len(genes) > 0, "Gene list is empty: check keep_methods against the labels printed above"

In [ ]:
import requests
try:
    r = requests.get("https://string-db.org/api/json/version", timeout=30)
    print(r.status_code, r.text)
except Exception as e:
    print("CANNOT REACH STRING:", type(e).__name__, e)

In [ ]:
import io, time

def string_post(method, **params):
    params.update(species=4932, caller_identity="mito_ppi_project")
    r = requests.post(f"https://string-db.org/api/tsv/{method}", data=params, timeout=300)
    if not r.ok:
        raise RuntimeError(f"STRING {method} failed ({r.status_code}): {r.text[:300]}")
    return pd.read_csv(io.StringIO(r.text), sep="\t")

mapped = string_post("get_string_ids", identifiers="\r".join(genes["orf"]), echo_query=1)
print(len(mapped), "of", len(genes), "genes recognised")
time.sleep(1)

raw = string_post("network", identifiers="\r".join(mapped["stringId"]),
                  required_score=700, network_type="physical").drop_duplicates()

edges = pd.DataFrame({
    "source": raw["stringId_A"].str.replace("4932.", "", regex=False),
    "target": raw["stringId_B"].str.replace("4932.", "", regex=False),
    "score":  raw["score"]})
edges.to_csv("mito_ppi_edges.tsv", sep="\t", index=False)
print(len(edges), "interactions")

In [ ]:
%pip install py4cytoscape
import py4cytoscape as p4c
p4c.cytoscape_ping()        # must print "You are connected to Cytoscape!"

edges["interaction"] = "pp"
in_net = set(edges["source"]) | set(edges["target"])
nodes = genes[genes["orf"].isin(in_net)].rename(columns={"orf": "id"})

p4c.create_network_from_data_frames(nodes=nodes, edges=edges,
                                    title="Yeast mitochondrial PPI",
                                    collection="SGD GO:0005739")
p4c.set_node_label_mapping("symbol")
p4c.layout_network("force-directed")
p4c.save_session("mito_ppi.cys")
print(len(nodes), "proteins,", len(edges), "interactions")

Just checking the amount of interaction according to confidence score and the type 

In [ ]:
for ntype in ["functional", "physical"]:
    for score in [400, 700]:
        n = len(string_post("network", identifiers="\r".join(mapped["stringId"]),
                            required_score=score, network_type=ntype).drop_duplicates())
        print(f"{ntype:10s} score>={score}: {n} interactions")
        time.sleep(1)

In [ ]:
import networkx as nx, matplotlib.pyplot as plt

G = nx.from_pandas_edgelist(edges, "source", "target", edge_attr="score")
G.remove_edges_from(nx.selfloop_edges(G))
comps = sorted(nx.connected_components(G), key=len, reverse=True)
GC = G.subgraph(comps[0])                     # largest connected component

summary = {
    "nodes": G.number_of_nodes(),
    "edges": G.number_of_edges(),
    "average degree": 2 * G.number_of_edges() / G.number_of_nodes(),
    "density": nx.density(G),
    "connected components": len(comps),
    "largest component (nodes)": len(GC),
    "average clustering coefficient": nx.average_clustering(G),
    "transitivity": nx.transitivity(G),
    "average shortest path (largest comp.)": nx.average_shortest_path_length(GC),
    "diameter (largest comp.)": nx.diameter(GC),
    "degree assortativity": nx.degree_assortativity_coefficient(G),
}
pd.Series(summary).round(4).to_csv("network_summary.tsv", sep="\t", header=False)
print(pd.Series(summary).round(4))

# Per-protein table
node_stats = pd.DataFrame({
    "degree": dict(G.degree()),
    "betweenness": nx.betweenness_centrality(G),
    "closeness": nx.closeness_centrality(G),
    "clustering": nx.clustering(G),
}).rename_axis("id").reset_index()
node_stats = node_stats.merge(genes.rename(columns={"orf": "id"}), on="id", how="left")
node_stats.sort_values("degree", ascending=False).to_csv("node_metrics.tsv", sep="\t", index=False)
print(node_stats.sort_values("degree", ascending=False).head(10))    # top hubs

# Degree distribution
node_stats["degree"].value_counts().sort_index().plot(style="o", loglog=True)
plt.xlabel("Degree (k)"); plt.ylabel("Number of proteins"); plt.title("Degree distribution")
plt.savefig("degree_distribution.png", dpi=200, bbox_inches="tight")

In [ ]:
p4c.load_table_data(node_stats, data_key_column="id", table_key_column="name")
p4c.set_node_size_mapping("degree", [1, 20, 80], [20, 50, 100])

In [ ]:
core = nx.k_core(G)                           # densest core of the network
sym = genes.set_index("orf")["symbol"]
print(len(core), "proteins and", core.number_of_edges(), "edges in the densest core")
print(sorted(sym.reindex(list(core)).tolist()))

print("median degree, whole network:", node_stats["degree"].median())

rest = G.copy()
rest.remove_nodes_from(list(core.nodes))
rest.remove_nodes_from(list(nx.isolates(rest)))
print("\nWithout the core:", rest.number_of_nodes(), "proteins,", rest.number_of_edges(), "edges")
print("average clustering:", round(nx.average_clustering(rest), 4))
print("transitivity:", round(nx.transitivity(rest), 4))
print("assortativity:", round(nx.degree_assortativity_coefficient(rest), 4))

# Bottleneck proteins: more informative than degree here
print(node_stats.sort_values("betweenness", ascending=False)
      .head(15)[["symbol", "degree", "betweenness"]])


In [ ]:
%run network_layout.py

In [ ]:
%run static_attack.py

In [ ]:
%run attack_views.py

In [ ]:
%run adaptive_attack

In [ ]:
%run random_failure.py

In [ ]:
%run random_fractions.py

In [ ]:
%run figures.py